[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/internshipinabook/mlops-internship-in-a-book/blob/main/notebooks/week4_communicating_failures_STARTER.ipynb)


[![Get the Book on Selar](https://img.shields.io/badge/Get%20the%20full%20book-Selar-1E7A34?style=for-the-badge)](https://selar.com/7m27877571)

# Week 4 -- Communicating System Failures (STARTER)
### The MLOps Internship | DataFlow Infrastructure

**Dataset:** CreditDefault-NG (UCI Credit Card Default)

**This week:** SLA definition, incident brief (Pyramid Principle), technical postmortem (5 Whys), action items

STARTER -- complete each exercise before moving on.


In [ ]:
# -- Colab Setup (skip if running locally) -------------------------
import os
if not os.path.exists('requirements.txt'):
    !git clone https://github.com/internshipinabook/mlops-internship-in-a-book.git mlops-book
    %cd mlops-book
    !pip install -r requirements.txt -q
os.makedirs('outputs', exist_ok=True)
os.makedirs('models',  exist_ok=True)
os.makedirs('datasets', exist_ok=True)
print('Setup complete.')


In [ ]:
# -- Reproducibility Seeds ------------------------------------------
# SEED=42 ensures every random operation gives the same result on every run.
# Set seeds BEFORE any data loading, model creation, or training.
import random, numpy as np
SEED = 42
random.seed(SEED)       # Python random
np.random.seed(SEED)    # NumPy random (used by sklearn)
print(f'Seeds fixed: {SEED}')


## Learning Objectives

- Define the SLA for the credit model: availability, latency, model quality floor, drift threshold
- Write a business incident brief for Farida in Pyramid Principle format
- Write a technical postmortem using the 5 Whys to arrive at a systemic process failure
- Write four action items that are specific, owned, dated, and verifiable
- Calculate the business impact of the Q4 drift in financial terms



---

## MONDAY -- "Defining the SLA"


An SLA without a model quality threshold is just an uptime agreement. The credit model has been running with a technically healthy SLA (99.9% uptime, p99 latency < 200ms) while systematically misclassifying high-limit customers. This week you add the missing terms: model quality (AUC floor), drift threshold (PSI limit), and fairness (approval rate stability).

Pause and Predict -- what is the minimum AUC you would accept for a production credit decision model that affects real lending decisions? Set the threshold before you see what the model currently achieves.


### Exercise 4.1 -- SLA definition

Define the credit model SLA: availability, latency, AUC threshold, PSI threshold, approval rate stability, and per-slice AUC. For each: threshold, measurement frequency, alert action.


In [ ]:
# Exercise 4.1: SLA definition
# ----------------------------------------------------------
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the end.

# YOUR CODE HERE


**Monday scaffold** (read and run, then adapt in exercises):


In [ ]:
# -- Monday: Defining the SLA --
SLA = {
    # Infrastructure SLAs -- already defined and monitored
    'availability': '99.9% uptime, measured monthly',
    'latency_p99':  '< 200ms prediction latency',
    'error_rate':   '< 0.1% HTTP 5xx error rate',

    # Model quality SLAs -- NEW, added after this incident
    'auc_threshold':   'val_AUC >= 0.80, measured weekly on holdout set',
    'approval_drift':  '< 5pp deviation from Q1 baseline approval rate',
    'psi_threshold':   'PSI < 0.25 for any top-10 predictive feature',
    'slice_threshold': 'Per-slice AUC >= 0.75 for all LIMIT_BAL quartiles',

    # Alerting
    'auc_alert':  'Page on-call if AUC < 0.80 for 2 consecutive weeks',
    'psi_alert':  'Page on-call if PSI > 0.25 for any feature',
}
print('SLA defined.')
print('None of the model quality SLAs existed before this incident.')
print('That is why the incident was silent for four months.')


### Expected Output (illustrative — this is a plain `print()`, so the output below is exact)

```
SLA defined.
None of the model quality SLAs existed before this incident.
That is why the incident was silent for four months.
```
The interesting artifact isn't the print statement — it's the `SLA` dict itself: notice it
took an incident to add the four model-quality entries. Infrastructure SLAs existed from day
one; model-quality SLAs were reactive. Week 9-10 build the monitoring that makes them
proactive instead.


### Monday Morning Moment

*Slack -- Monday, 9:00am.*

**Farida Usman:** I need to brief the board on Friday. What do I tell them about the model?

**Temi Adeyemi:** That the model is working reliably for the majority of customers but has a known limitation for high-limit accounts that we are actively resolving. A retrained model will be in production by July 6th.

**Farida Usman:** Is that statement accurate?

**Temi Adeyemi:** Yes. Model AUC on standard-limit customers is 0.88. For high-limit customers it is 0.67. The timeline is confirmed.

**Dr. Emeka Obi:** Write that as the executive summary. That is exactly the right level of detail for the board.

**Sola Fashola:** Do not say 'AUC' in the business document. Say 'prediction accuracy' or 'prediction reliability'.




---

## TUESDAY -- "The Business Incident Brief"


Pyramid Principle: answer first, evidence second, methodology last. Farida reads the first paragraph and decides whether the situation is under control. The first paragraph must answer three questions: what happened, what the impact is, and what is being done. Everything else is evidence for those three statements.


### Exercise 4.2 -- Business brief

Write the incident brief for Farida in Pyramid Principle format. Test: read it to someone without ML background.


In [ ]:
# Exercise 4.2: Business brief
# ----------------------------------------------------------
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the end.

# YOUR CODE HERE


**Tuesday scaffold** (read and run, then adapt in exercises):


In [ ]:
# -- Tuesday: The Business Incident Brief --
# Business brief structure (Pyramid Principle):
# [STATUS: AMBER -- Under active remediation]
#
# EXECUTIVE SUMMARY (2 sentences -- write these first, before anything else):
# [What happened -- one sentence, no jargon]
# [Impact and remediation -- one sentence]
#
# WHAT HAPPENED (3 bullets, plain English, no PSI values):
# BUSINESS IMPACT (in customer counts and financial terms):
# WHAT WE ARE DOING (3 actions, each with owner and date):
# WHAT WE NEED FROM YOU (one specific ask from Farida):


### Expected Output (illustrative)

No code output — Tuesday's deliverable is the written business brief. The one non-negotiable
rule from this week's checklist: the executive summary is the *first* paragraph, and it
contains zero PSI values, KS statistics, or AUC numbers. If Farida has to ask "what does this
mean for our customers," the brief has failed regardless of its technical accuracy.



---

## WEDNESDAY -- "The Technical Postmortem: 5 Whys"


5 Whys drills from symptom to systemic root cause. The fifth why must always be a process failure -- not a human error, not a technical limitation. 'The engineer forgot to set up monitoring' is blame. 'There was no monitoring requirement in the deployment checklist' is a systemic finding that generates a fixable action item.


### Exercise 4.3 -- 5 Whys

Write the 5 Whys chain. Why 5 must be a systemic process failure. If it is a human error or technical limitation, keep drilling.


In [ ]:
# Exercise 4.3: 5 Whys
# ----------------------------------------------------------
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the end.

# YOUR CODE HERE


**Wednesday scaffold** (read and run, then adapt in exercises):


In [ ]:
# -- Wednesday: The Technical Postmortem: 5 Whys --
whys = [
    ('1', 'Why did high-risk customers receive incorrect approval decisions?',
     'The model underestimated default probability for high-limit customers.'),
    ('2', 'Why were high-limit customer estimates unreliable?',
     'High-limit customers were underrepresented in the Q1 training data (<3%).'),
    ('3', 'Why were high-limit customers underrepresented in training?',
     'The premium account segment was introduced by the bureau after Q1 data was collected.'),
    ('4', 'Why was the model not retrained when the new segment was introduced?',
     'There was no process requiring retraining when new customer segments entered the pipeline.'),
    ('5', 'Why was there no such process?',
     'Model deployment did not include a data pipeline change review gate.'),
]
for num, q, a in whys:
    print(f'Why {num}: {q}')
    print(f'  -> {a}')


### Expected Output (exact — pure `print()` loop, no external dependency)

```
Why 1: Why did high-risk customers receive incorrect approval decisions?
  -> The model underestimated default probability for high-limit customers.
Why 2: Why were high-limit customer estimates unreliable?
  -> High-limit customers were underrepresented in the Q1 training data (<3%).
Why 3: Why were high-limit customers underrepresented in training?
  -> The premium account segment was introduced by the bureau after Q1 data was collected.
Why 4: Why was the model not retrained when the new segment was introduced?
  -> There was no process requiring retraining when new customer segments entered the pipeline.
Why 5: Why was there no such process?
  -> Model deployment did not include a data pipeline change review gate.
```
Notice Why 5 lands on a *process* failure, not a technical one — that's what makes a 5 Whys
chain good. A chain that bottoms out at "the code had a bug" usually stopped one level too
early.



---

## THURSDAY -- "Action Items and Owners"


Action items are the only postmortem output that produces change. Each must be: specific (exactly what will be built), owned (one named person), dated (a concrete deadline), and verifiable (a criterion that confirms it is done). Missing any of these four properties guarantees the action item will not be completed.


### Exercise 4.4 -- Action items with done_when

Write four action items. Each needs: specific action, one owner, due date, and verifiable done_when criterion. One addresses the detection gap. One addresses the root cause.


In [ ]:
# Exercise 4.4: Action items with done_when
# ----------------------------------------------------------
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the end.

# YOUR CODE HERE


**Thursday scaffold** (read and run, then adapt in exercises):


In [ ]:
# -- Thursday: Action Items and Owners --
action_items = [
    {'id': 'AI-01', 'owner': 'Temi Adeyemi', 'due': '2026-06-22',
     'action': 'Deploy model quality SLA monitoring to Grafana',
     'done_when': 'PSI and AUC alerts tested by deliberately triggering them'},
    {'id': 'AI-02', 'owner': 'Sola Fashola', 'due': '2026-06-29',
     'action': 'Add data pipeline change review gate to ETL deployment process',
     'done_when': 'ETL PRs require ML team approval for any schema changes'},
    {'id': 'AI-03', 'owner': 'Temi Adeyemi', 'due': '2026-07-06',
     'action': 'Retrain credit model with Q2-Q4 data including premium segment',
     'done_when': 'New model in Production, AUC >= 0.85 on all LIMIT_BAL quartiles'},
    {'id': 'AI-04', 'owner': 'Farida Usman', 'due': '2026-06-15',
     'action': 'Manual review of 1,840 flagged Q4 decisions',
     'done_when': 'All 1,840 reviewed and outcomes logged'},
]
for ai in action_items:
    print(f"[{ai['id']}] {ai['action']} | {ai['owner']} | Due: {ai['due']}")


### Expected Output (exact — pure `print()` loop, no external dependency)

```
[AI-01] Deploy model quality SLA monitoring to Grafana | Temi Adeyemi | Due: 2026-06-22
[AI-02] Add data pipeline change review gate to ETL deployment process | Sola Fashola | Due: 2026-06-29
[AI-03] Retrain credit model with Q2-Q4 data including premium segment | Temi Adeyemi | Due: 2026-07-06
[AI-04] Manual review of 1,840 flagged Q4 decisions | Farida Usman | Due: 2026-06-15
```
Every action item has an owner, a date, and a `done_when` that is checkable by someone who
wasn't in the incident review — that third property is what separates an action item from a
good intention.



---

## FRIDAY -- "The Friday Build: Both Documents"


Deliver both documents today. Business brief: one page. Technical postmortem: two pages maximum. Test each: give the business brief to someone without ML background -- they must be able to answer 'what happened, are we at risk, and what is being done' after reading the first paragraph.


**Friday scaffold** (read and run, then adapt in exercises):


In [ ]:
# -- Friday: The Friday Build: Both Documents --
# Business brief checklist:
# - No PSI values, KS statistics, or AUC numbers in the summary
# - Impact in customer counts and financial terms
# - Executive summary is the FIRST paragraph
# - Three action items each have an owner and a date

# Technical postmortem checklist:
# - Timeline section with specific dates
# - 5 Whys chain -- Why 5 ends in a systemic process failure
# - Four action items with owners, due dates, and done_when criteria
# - 'What went well' section (three things that limited the damage)


### Expected Output (illustrative)

No code output — Friday delivers two documents. Use the checklist in the cell above them as a
literal pass/fail gate before you consider either document finished; "I think it's thorough" is
not the same as "every box is checked."


### Friday Workplace Moment

*DataFlow Infrastructure -- Friday standup.*

**Dr. Emeka Obi:** Business brief. Read me the executive summary.

**Temi Adeyemi:** 'The credit model's prediction quality degraded between February and June due to a new high-value customer segment introduced by the credit bureau. Approximately 1,840 Q4 decisions for this segment are at elevated reliability risk and have been flagged for manual review. A fully retrained model will be in production by July 6th.'

**Dr. Emeka Obi:** Two sentences. Good. What does Why 5 say is the systemic root cause?

**Temi Adeyemi:** 'Model deployment did not include a data pipeline change review gate.' That is the process failure that generates AI-02.

**Sola Fashola:** AI-02 is mine. Gate will be in place by June 29th.



## YOUR WEEK 4 CHECKLIST

Check each item before moving on.

- [ ] The SLA includes a model quality threshold (AUC floor), not just infrastructure SLAs.
- [ ] My business brief passes the non-ML-reader test: they can answer what happened, the risk, and the remediation.
- [ ] The 5 Whys chain ends in a systemic process failure, not a human error.
- [ ] All action items have a single owner, a due date, and a verifiable done_when criterion.
- [ ] I can explain the difference between a business brief and a technical postmortem in one sentence.


## Challenge Task

> **Core path:** Calculate the financial impact: given estimated default rate for misclassified segment, average credit limit, and number of affected decisions, estimate expected credit loss. Build sensitivity table for optimistic, realistic, and pessimistic scenarios.

> **Advanced path:** Write the 'what went well' section. Find three things that worked correctly during this incident and prevented it from being worse.


## Common Mistakes

**Postmortem that assigns blame to a person:** Postmortems identify systemic failures. 'Engineer did not set up monitoring' is blame. 'No monitoring requirement existed in the deployment checklist' generates a fixable action item.

**Sending the technical postmortem to a business stakeholder:** PSI values and 5 Whys chains in a business document will be misread or ignored. Two documents, two audiences.

**Action items without done_when criteria:** An action item without a verifiable completion criterion will live in the backlog forever.



---

**The MLOps Internship** -- Book 4 of 9, InternshipInABook™

Dataset: CreditDefault-NG | Company: DataFlow Infrastructure, Lagos/Abuja

[internshipinabook.com](https://internshipinabook.com)


📘 **Get the complete illustrated book (all 13 weeks, DOCX + EPUB):** [https://selar.com/7m27877571](https://selar.com/7m27877571)